# QuackWeave usage patterns on the LSQB SF1 Ladybug graph

**Latest release comparison.** On `18807b8`, the table-backed watchlist takes 9.73 ms (previously 380.19 ms), and the full aggregate takes 138.28 ms (previously 344.72 ms). The [follow-up measurements](#follow-up-on-the-table-watchlist-and-full-aggregate) use `a2098d7` as the baseline.

The earlier comparison below measures QuackWeave `a2098d7` with `ecfcd8e`, the release preceding read-only database handle reuse and predicate pushdown. It also runs equivalent graph queries directly in Ladybug 0.22.3. The graph contains 3,955,790 nodes and 23,697,561 relationships. The measured traversal reads the 3,809,607 message-creator edges on that graph.

The extension runs inside a real DuckDB 2.0.1 process. The DuckDB cells below analyze the samples from those processes. Open the Explore tab on the chart cells to compare engines.

### Findings on this graph

All 14 data-returning workloads got faster in every paired round. Their results matched direct Ladybug. All 15 pooled p95 measurements fell in the mixed workload. A dedicated planning-only test exposed a small regression for the literal join. Its median `EXPLAIN` latency rose from 0.460 to 0.548 ms (+0.088 ms, +19%) over 800 observations per build. The current build was slower in every one of the eight process pairs. Its planning p95 rose from 0.592 to 0.682 ms. The median mixed-workload process peak was 742 MiB on the current build and 750 MiB on the baseline.

For the same five author IDs, the literal watchlist fell from 416.84 to 8.39 ms. A temporary DuckDB table holding those IDs took 340.86 ms on the current build. The plans show 23,316 and 3,809,607 rows crossing into DuckDB, respectively. The unfiltered SQL aggregate took 309.32 ms; direct Ladybug computed the same aggregate in 128.38 ms. The host-table join and the full scan both improved against the previous release, but still account for the largest absolute latencies.

## Measurement and regression criteria

Four paired rounds alternate which extension runs first. Each engine gets a fresh process and connection per round. After a first point lookup, each workload runs once to warm up and five times for measurement. Workload order changes with a recorded deterministic seed. The 20 observations per extension and workload time execute through Arrow fetch. Validation, dialect selection and the separate `EXPLAIN ANALYZE` profiles are outside that interval. The Arrow reuse case includes importing the result into a second DuckDB connection and running the host aggregate. The native reference times the equivalent graph answer only.

A warm median increase over **both 10% and 1 ms** is the practical regression screen for this run. Inspect the paired rounds and p95 separately. Direct Ladybug provides a correctness oracle and performance reference. Every timed data query must match its native result, including duplicates and any specified ordering. P9 is checked by hashing the database before and after. P12 checks that the plan contains the graph operator. A fresh process still benefits from the operating system’s file cache.

In [2]:
WITH report AS (SELECT content::JSON AS doc FROM read_text(['usage_regression.json', 'benchmarks/lsqb_sf1/usage_regression.json']) LIMIT 1), rows AS (
  SELECT unnest(from_json(doc->'summary', '[{"workload":"VARCHAR","pattern":"VARCHAR","engine":"VARCHAR","n":"INTEGER","p50_ms":"DOUBLE","p95_ms":"DOUBLE","cpu_ms":"DOUBLE","provider_rows":["BIGINT"]}]')) AS s FROM report
), measurements AS (SELECT s.* FROM rows)
SELECT workload, any_value(pattern) AS pattern,
       round(max(p50_ms) FILTER (engine='baseline'),2) AS baseline_ms,
       round(max(p50_ms) FILTER (engine='current'),2) AS current_ms,
       round(max(p50_ms) FILTER (engine='ladybug'),2) AS native_ms,
       round(max(p95_ms) FILTER (engine='baseline'),2) AS baseline_p95_ms,
       round(max(p95_ms) FILTER (engine='current'),2) AS current_p95_ms,
       round(max(p50_ms) FILTER (engine='baseline') / max(p50_ms) FILTER (engine='current'),1) AS speedup
FROM measurements GROUP BY workload ORDER BY speedup DESC

workload,pattern,baseline_ms,current_ms,native_ms,baseline_p95_ms,current_p95_ms,speedup
graph_table_equality,P2,411.71,3.5,2.83,476.1,4.34,117.8
view_aggregate,P6/P7,407.74,3.65,2.47,430.45,4.82,111.8
view_self_join,P4/P7,748.53,6.96,5.97,947.03,9.69,107.6
cte_order_limit,P2/P3,363.07,3.66,2.55,403.63,4.3,99.3
graph_table_in,P2/P6,378.98,7.44,4.34,435.14,9.95,50.9
literal_watchlist,P3/P10,416.84,8.39,4.34,504.28,10.33,49.7
view_exists,P5/P7,383.89,8.52,1.69,432.36,10.52,45.0
cypher_fanout,P1,40.99,2.71,2.83,53.04,4.05,15.1
gql_lookup,P1,33.61,2.49,0.86,42.13,3.38,13.5
prepared_match,P8,34.3,2.58,0.86,42.03,3.45,13.3


## Selective queries

The chart compares current QuackWeave and direct Ladybug for queries that return a single key, a single author’s messages, or the selected watchlist groups. GQL and Cypher fanout use the same native query. For a hybrid query, the native reference computes the equivalent answer entirely inside Ladybug.

In [3]:
WITH report AS (SELECT content::JSON AS doc FROM read_text(['usage_regression.json', 'benchmarks/lsqb_sf1/usage_regression.json']) LIMIT 1), rows AS (
  SELECT unnest(from_json(doc->'summary', '[{"workload":"VARCHAR","pattern":"VARCHAR","engine":"VARCHAR","n":"INTEGER","p50_ms":"DOUBLE","p95_ms":"DOUBLE","cpu_ms":"DOUBLE","provider_rows":["BIGINT"]}]')) AS s FROM report
), measurements AS (SELECT s.* FROM rows)
SELECT workload, engine, round(p50_ms,2) AS p50_ms FROM measurements
WHERE engine IN ('current','ladybug') AND workload IN ('gql_lookup','gql_fanout','cypher_fanout','graph_table_equality','literal_watchlist','graph_table_in')
ORDER BY workload,engine

workload,engine,p50_ms
cypher_fanout,current,2.71
cypher_fanout,ladybug,2.83
gql_fanout,current,5.1
gql_fanout,ladybug,2.83
gql_lookup,current,2.49
gql_lookup,ladybug,0.86
graph_table_equality,current,3.5
graph_table_equality,ladybug,2.83
graph_table_in,current,7.44
graph_table_in,ladybug,4.34


## Where rows cross into DuckDB

The next chart compares two ways to supply identical watchlist keys and an unfiltered scan. The table underneath counts rows at every QuackWeave scan in a separate analyzed plan. A self-join has two scans. These are rows handed to DuckDB by the provider, distinct from rows read internally by Ladybug.

In [4]:
WITH report AS (SELECT content::JSON AS doc FROM read_text(['usage_regression.json', 'benchmarks/lsqb_sf1/usage_regression.json']) LIMIT 1), rows AS (
  SELECT unnest(from_json(doc->'summary', '[{"workload":"VARCHAR","pattern":"VARCHAR","engine":"VARCHAR","n":"INTEGER","p50_ms":"DOUBLE","p95_ms":"DOUBLE","cpu_ms":"DOUBLE","provider_rows":["BIGINT"]}]')) AS s FROM report
), measurements AS (SELECT s.* FROM rows)
SELECT workload, engine, round(p50_ms,2) AS p50_ms FROM measurements
WHERE workload IN ('literal_watchlist','table_watchlist','full_scan') ORDER BY workload, engine

workload,engine,p50_ms
full_scan,baseline,395.08
full_scan,current,309.32
full_scan,ladybug,128.38
literal_watchlist,baseline,416.84
literal_watchlist,current,8.39
literal_watchlist,ladybug,4.34
table_watchlist,baseline,432.82
table_watchlist,current,340.86
table_watchlist,ladybug,4.34


In [5]:
WITH report AS (SELECT content::JSON AS doc FROM read_text(['usage_regression.json', 'benchmarks/lsqb_sf1/usage_regression.json']) LIMIT 1), rows AS (
  SELECT unnest(from_json(doc->'summary', '[{"workload":"VARCHAR","pattern":"VARCHAR","engine":"VARCHAR","n":"INTEGER","p50_ms":"DOUBLE","p95_ms":"DOUBLE","cpu_ms":"DOUBLE","provider_rows":["BIGINT"]}]')) AS s FROM report
), measurements AS (SELECT s.* FROM rows)
SELECT workload, max(provider_rows) FILTER (engine='baseline') AS baseline_scan_rows,
       max(provider_rows) FILTER (engine='current') AS current_scan_rows
FROM measurements WHERE provider_rows IS NOT NULL GROUP BY workload ORDER BY workload

workload,baseline_scan_rows,current_scan_rows
cte_order_limit,[3809607],[6465]
full_scan,[3809607],[3809607]
graph_table_equality,[3809607],[6465]
graph_table_in,[3809607],[23316]
literal_watchlist,[3809607],[23316]
table_watchlist,[3809607],[3809607]
view_aggregate,[3809607],[6465]
view_exists,[3809607],[23316]
view_self_join,"[3809607, 3809607]","[6465, 6465]"


## Fresh instances and process memory

Setup includes connecting to DuckDB, checking its build, loading the extension and selecting the graph. For direct Ladybug, setup opens the database and connection. The first point lookup is timed separately before workload warm-up. Peak RSS is the process high-water mark for the mixed workload, including the Python client, Arrow, engine caches and host joins. It is sampled before plan profiling.

In [6]:
WITH report AS (SELECT content::JSON AS doc FROM read_text(['usage_regression.json', 'benchmarks/lsqb_sf1/usage_regression.json']) LIMIT 1), blocks AS (
  SELECT unnest(from_json(doc->'blocks', '[{"engine":"VARCHAR","setup_ms":"DOUBLE","first_point_ms":"DOUBLE","peak_rss_mib":"DOUBLE"}]')) AS b FROM report
)
SELECT b.engine AS engine, count(*) AS processes, round(median(b.setup_ms),2) AS setup_ms,
       round(median(b.first_point_ms),2) AS first_lookup_ms,
       round(median(b.setup_ms+b.first_point_ms),2) AS setup_and_first_ms,
       round(median(b.peak_rss_mib),1) AS median_peak_mib, round(max(b.peak_rss_mib),1) AS max_peak_mib
FROM blocks GROUP BY b.engine ORDER BY b.engine

engine,processes,setup_ms,first_lookup_ms,setup_and_first_ms,median_peak_mib,max_peak_mib
baseline,4,83.14,44.72,127.86,749.8,764.0
current,4,83.96,29.08,113.05,742.0,748.4
ladybug,4,49.28,8.92,58.2,528.0,530.0


## Variation across paired rounds

Each round compares two fresh QuackWeave processes using the same seed and workload order. Positive deltas indicate the current build took longer. The last two columns count rounds where the current median increased at all and where it crossed both regression thresholds. The p95 in the first table uses the nearest-rank observation.

In [7]:
WITH report AS (SELECT content::JSON AS doc FROM read_text(['usage_regression.json', 'benchmarks/lsqb_sf1/usage_regression.json']) LIMIT 1), timings AS (
  SELECT b.value->>'engine' AS engine, (b.value->>'block')::INT AS block_id,
         w.key AS workload, (t.value->>0)::DOUBLE AS wall_ms, (t.value->>1)::DOUBLE AS cpu_ms
  FROM report, json_each(doc->'blocks') b, json_each(b.value->'samples') w, json_each(w.value) t
  WHERE b.value->>'engine' IN ('baseline','current')
), paired AS (
  SELECT workload, block_id, median(wall_ms) FILTER (engine='baseline') AS old_ms,
         median(wall_ms) FILTER (engine='current') AS new_ms,
         median(cpu_ms) FILTER (engine='current') AS new_cpu
  FROM timings GROUP BY workload, block_id
)
SELECT workload, round(min(new_ms-old_ms),2) AS min_delta_ms,
       round(max(new_ms-old_ms),2) AS max_delta_ms, round(median(new_cpu),2) AS current_cpu_ms,
       count(*) FILTER (new_ms>old_ms) AS slower_rounds,
       count(*) FILTER (new_ms>1.10*old_ms AND new_ms-old_ms>1) AS flagged_rounds
FROM paired GROUP BY workload ORDER BY max_delta_ms DESC

workload,min_delta_ms,max_delta_ms,current_cpu_ms,slower_rounds,flagged_rounds
explain_only,-0.25,0.08,1.06,2,0
table_watchlist,-104.0,-24.48,989.01,0,0
gql_lookup,-37.15,-26.25,2.92,0,0
prepared_match,-32.11,-26.99,2.91,0,0
cypher_fanout,-48.0,-32.64,4.69,0,0
arrow_reuse,-41.09,-32.94,9.09,0,0
gql_fanout,-39.11,-35.22,7.79,0,0
full_scan,-113.8,-49.89,961.15,0,0
view_exists,-418.75,-320.12,20.65,0,0
view_aggregate,-417.98,-321.02,4.0,0,0


### Targeted repeat

A second run uses four more process pairs and six warm samples per workload per process. It checks planning jitter, point lookup, and the cost of supplying the same watchlist as literals or a DuckDB table. The native watchlist timings are for a graph query that already has the five keys.

In [8]:
WITH report AS (SELECT content::JSON AS doc FROM read_text(['usage_regression_repeat.json', 'benchmarks/lsqb_sf1/usage_regression_repeat.json']) LIMIT 1), rows AS (
  SELECT unnest(from_json(doc->'summary', '[{"workload":"VARCHAR","pattern":"VARCHAR","engine":"VARCHAR","n":"INTEGER","p50_ms":"DOUBLE","p95_ms":"DOUBLE","cpu_ms":"DOUBLE","provider_rows":["BIGINT"]}]')) AS s FROM report
), measurements AS (SELECT s.* FROM rows)
SELECT workload, any_value(n) AS warm_samples_per_engine,
       round(max(p50_ms) FILTER (engine='baseline'),2) AS baseline_ms,
       round(max(p50_ms) FILTER (engine='current'),2) AS current_ms,
       round(max(p50_ms) FILTER (engine='ladybug'),2) AS native_ms,
       round(max(p95_ms) FILTER (engine='baseline'),2) AS baseline_p95_ms,
       round(max(p95_ms) FILTER (engine='current'),2) AS current_p95_ms
FROM measurements GROUP BY workload ORDER BY workload

workload,warm_samples_per_engine,baseline_ms,current_ms,native_ms,baseline_p95_ms,current_p95_ms
explain_only,24,1.01,1.12,NULL,1.28,1.41
gql_lookup,24,34.14,3.13,0.82,48.5,3.63
literal_watchlist,24,432.07,10.07,3.84,525.24,11.73
table_watchlist,24,427.74,395.4,3.84,587.75,417.3


### Planning latency in isolation

The four-workload repeat put current `EXPLAIN` at 1.12 ms against 1.01 ms for baseline. The following check removes the other timed workloads. Each of eight new process pairs plans the literal-watchlist query 100 times after warm-up, using `EXPLAIN` without `ANALYZE`. This isolates DuckDB planning, plan serialization and fetching the plan over ADBC.

In [9]:
WITH report AS (SELECT content::JSON AS doc FROM read_text(['usage_regression_planning.json', 'benchmarks/lsqb_sf1/usage_regression_planning.json']) LIMIT 1), samples AS (
  SELECT b.value->>'engine' AS engine, (b.value->>'block')::INT AS block_id, (t.value->>0)::DOUBLE AS ms
  FROM report, json_each(doc->'blocks') b, json_each(b.value->'samples'->'explain_only') t
), paired AS (
  SELECT block_id, median(ms) FILTER (engine='baseline') AS old_ms,
         median(ms) FILTER (engine='current') AS new_ms FROM samples GROUP BY block_id
)
SELECT count(*) FILTER (engine='current') AS observations_per_build,
       round(median(ms) FILTER (engine='baseline'),3) AS baseline_p50_ms,
       round(median(ms) FILTER (engine='current'),3) AS current_p50_ms,
       round(quantile_disc(ms, 0.95) FILTER (engine='baseline'),3) AS baseline_p95_ms,
       round(quantile_disc(ms, 0.95) FILTER (engine='current'),3) AS current_p95_ms,
       (SELECT count(*) FROM paired WHERE new_ms>old_ms) AS slower_pairs
FROM samples

observations_per_build,baseline_p50_ms,current_p50_ms,baseline_p95_ms,current_p95_ms,slower_pairs
800,0.46,0.548,0.592,0.682,8


## Reproduce the measurement

From the repository root, run these commands with the pinned release packages and the published SF1 file in their existing build paths. Re-execute the SQL cells to refresh the analysis. The JSON contains per-sample wall and CPU times, exact statements, native equivalents, result signatures, per-process measurements, provider cardinalities and binary hashes.

```sh
build/usage-client-venv/bin/python -B benchmarks/lsqb_sf1/usage_regression.py
build/usage-client-venv/bin/python -B benchmarks/lsqb_sf1/usage_regression.py \
  --blocks 4 --runs 6 --cases explain_only,gql_lookup,literal_watchlist,table_watchlist \
  --out benchmarks/lsqb_sf1/usage_regression_repeat.json
build/usage-client-venv/bin/python -B benchmarks/lsqb_sf1/usage_regression.py \
  --blocks 8 --runs 100 --cases explain_only \
  --out benchmarks/lsqb_sf1/usage_regression_planning.json
```

The timings describe one macOS arm64 host with one active client. They cover interactive reads against a warm filesystem cache. Cold disk, concurrent clients, different graph sizes and other key distributions require separate measurements. [LSQB SF1 dataset](https://huggingface.co/datasets/ladybugdb/lsqb-sf1/blob/8283a51fa16c4d9fbc3c53ebde66252f7a41f3f6/lbdb/sf1.lbdb)

## Follow-up on the table watchlist and full aggregate

This comparison uses the archived `a2098d7` package as its baseline. Both extension builds execute the original SQL on the same LSQB SF1 file. The direct Ladybug reference runs a native aggregate or a query with the same five author IDs. Each extension timing covers execution and fetching the complete result over ADBC. The analyzed plans count the rows handed to DuckDB by each QuackWeave scan.

To repeat from the repository root after building the release, run

```sh
build/usage-client-venv/bin/python -B benchmarks/lsqb_sf1/usage_regression.py \
  --verify-pushdown \
  --baseline build/perf/lsqb-sf1/baseline-a2098d7/quack_weave.duckdb_extension \
  --candidate build/native/release/extension/quack_weave/quack_weave.duckdb_extension \
  --out benchmarks/lsqb_sf1/usage_pushdown.json
```

In [10]:
WITH report AS (SELECT content::JSON AS doc FROM read_text(['usage_pushdown.json','benchmarks/lsqb_sf1/usage_pushdown.json']) LIMIT 1)
SELECT s.value->>'workload' AS workload, s.value->>'engine' AS engine,
       round((s.value->>'p50_ms')::DOUBLE, 2) AS p50_ms,
       round((s.value->>'p95_ms')::DOUBLE, 2) AS p95_ms,
       (s.value->>'n')::INT AS samples, s.value->'provider_rows' AS graph_scan_rows
FROM report, json_each(doc->'summary') s
WHERE s.value->>'workload' IN ('table_watchlist', 'full_scan', 'literal_watchlist')
ORDER BY workload, engine

workload,engine,p50_ms,p95_ms,samples,graph_scan_rows
full_scan,baseline,344.72,397.57,20,[3809607]
full_scan,current,138.28,148.04,20,[1]
full_scan,ladybug,136.33,155.34,20,null
literal_watchlist,baseline,9.18,10.42,20,[23316]
literal_watchlist,current,10.12,11.33,20,[23316]
literal_watchlist,ladybug,4.53,5.12,20,null
table_watchlist,baseline,380.19,420.29,20,[3809607]
table_watchlist,current,9.73,12.48,20,[23316]
table_watchlist,ladybug,4.53,5.12,20,null


### Follow-up measurements

The same SQL against the same SF1 file took **9.73 ms** for the table-backed watchlist and **138.28 ms** for the full aggregate on `18807b8`. The archived `a2098d7` took **380.19 ms** and **344.72 ms**, respectively. The provider transferred 23,316 matching edges for the watchlist and one result row for the aggregate. Direct Ladybug computed the aggregate in 136.33 ms and the four watchlist groups in 4.53 ms with the author IDs already available. All returned values matched. The source file retained the same SHA-256.

The mixed run showed a 1.06 ms increase for an existing graph-local `IN` filter. The control experiment repeats that case alongside five short queries, with eight alternating process pairs and twelve timed executions per workload and build in each pair.

In [11]:
WITH report AS (
  SELECT content::JSON AS doc FROM read_text(['usage_pushdown_controls.json','benchmarks/lsqb_sf1/usage_pushdown_controls.json']) LIMIT 1
), timings AS (
  SELECT s.value->>'workload' AS workload, s.value->>'engine' AS engine,
         (s.value->>'p50_ms')::DOUBLE AS ms, (s.value->>'n')::INTEGER AS n
  FROM report, json_each(doc->'summary') s
), comparisons AS (
  SELECT workload, max(n) AS n, max(ms) FILTER (engine='baseline') AS old_ms,
         max(ms) FILTER (engine='current') AS new_ms FROM timings GROUP BY workload
)
SELECT workload, n AS samples_per_build, round(old_ms,2) AS baseline_ms, round(new_ms,2) AS current_ms,
       round(new_ms-old_ms,2) AS delta_ms, round(100*(new_ms/old_ms-1),1) AS delta_pct
FROM comparisons ORDER BY delta_ms DESC

workload,samples_per_build,baseline_ms,current_ms,delta_ms,delta_pct
literal_watchlist,96,8.04,8.18,0.14,1.7
gql_fanout,96,5.06,5.18,0.11,2.2
view_exists,96,8.19,8.29,0.11,1.3
prepared_match,96,2.53,2.56,0.03,1.4
explain_only,96,0.83,0.84,0.02,2.0
graph_table_in,96,7.59,7.59,0.01,0.1


The isolated control run put the graph-local `IN` query at 7.59 ms on both builds (0.01 ms difference before rounding). The other five control medians increased by 0.02–0.14 ms. All were below the benchmark's 1 ms absolute regression criterion. The median peak resident memory across the main mixed-workload processes fell from 743 to 549 MiB.

To repeat the control run from the repository root, use

```sh
build/usage-client-venv/bin/python -B benchmarks/lsqb_sf1/usage_regression.py \
  --baseline build/perf/lsqb-sf1/baseline-a2098d7/quack_weave.duckdb_extension \
  --candidate build/native/release/extension/quack_weave/quack_weave.duckdb_extension \
  --cases graph_table_in,literal_watchlist,view_exists,gql_fanout,prepared_match,explain_only \
  --blocks 8 --runs 12 --out benchmarks/lsqb_sf1/usage_pushdown_controls.json
```

The table watchlist fast path uses an exact runtime key filter from DuckDB's hash join. Plans that cannot supply that filter may still read the full graph relation. The aggregate path covers eligible ungrouped counts and integer sums. SQLLogic tests cover fallback shapes, empty inputs, NULLs, signed 128-bit sums, and explicit execution limits. These latency measurements describe one macOS arm64 host with one active client and a warm filesystem cache.